# The holdout, scored once

`demand holdout` ran once (`reports/holdout.json`). This notebook only reads its saved results:
8 weekly origins from 15 October to 3 December 2011, each refitted on everything before it,
scored on the next 14 open days up to the last day of data, 9 December. The model was picked on
the backtest (decision 31) before this was run.

In [1]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir("..")

from demand import data
from demand import intervals as iv
from demand.backtest import origins
from demand.compare import paired_difference
from demand.plots import COLORS, GRID, INK, LABELS, MUTED
from demand.series import TOTAL

processed = data.data_dir() / "processed"
holdout = pd.read_parquet(processed / "holdout_results.parquet")
report = json.loads(Path("reports/holdout.json").read_text())
backtest = json.loads(Path("reports/backtest.json").read_text())
products, total = holdout[holdout["series"] != TOTAL], holdout[holdout["series"] == TOTAL]
report["origins"]

['2011-10-15',
 '2011-10-22',
 '2011-10-29',
 '2011-11-05',
 '2011-11-12',
 '2011-11-19',
 '2011-11-26',
 '2011-12-03']

In [2]:
rows = {}
for model, s in report["models"].items():
    rows[model] = {
        "backtest products wMAPE": backtest[model]["products"]["wmape"],
        "holdout products wMAPE": s["products"]["wmape"],
        "holdout products bias": s["products"]["bias"],
        "holdout total wMAPE": s["total"]["wmape"],
        "holdout total bias": s["total"]["bias"],
    }
pd.DataFrame(rows).T.round(3)

,backtest products wMAPE,holdout products wMAPE,holdout products bias,holdout total wMAPE,holdout total bias
lightgbm_no_christmas,0.881,0.794,-0.109,0.236,-0.015
ets,0.939,0.830,-0.075,0.223,-0.063
weekday_mean_4,0.998,0.901,-0.105,0.244,-0.093


The error is *lower* on the holdout than in the backtest, for every model. I expected the
opposite. Peak-season days are bigger, and bigger numbers are relatively steadier, so wMAPE
falls even though the forecasts lag. The bias shows the lag.

In [3]:
pick = "lightgbm_no_christmas"
pd.DataFrame(
    {
        (level, ref): paired_difference(part, pick, ref)
        for level, part in (("products", products), ("total", total))
        for ref in ("ets", "weekday_mean_4")
    }
).T.round(3)

difference    low   high  origins_a_better  origins
products ets                 -0.036 -0.046 -0.023               7.0      8.0
         weekday_mean_4      -0.107 -0.119 -0.095               8.0      8.0
total    ets                  0.013 -0.004  0.031               2.0      8.0
         weekday_mean_4      -0.009 -0.034  0.023               4.0      8.0

In [4]:
by_month = products.assign(month=products["origin"].dt.month).groupby(["model", "month"])
(by_month["forecast"].sum() / by_month["actual"].sum() - 1).unstack().round(3)

month,10,11,12
model,,,
ets,-0.051,-0.107,0.094
lightgbm_no_christmas,-0.085,-0.153,0.172
weekday_mean_4,-0.070,-0.156,0.173


By origin (chart below): under-forecasting from late October to mid-November while sales climb
(-29% and -33% for the 29 October and 5 November origins), over-forecasting once they turn
(+17% for 3 December). The first origin, 15 October, was 14% over: the September step had
already lifted the recent level. The model follows the level a few weeks late; it has no notion
of the season beyond what recent weeks show, and the one feature that tried (weeks to Christmas)
was dropped on Day 8 because, learned from a single autumn, it memorised rather than generalised.

### Do the 80% intervals hold?

In [5]:
wide = pd.read_parquet(processed / "daily_units.parquet")
staggered = pd.read_parquet(processed / "backtest_staggered.parquet")
staggered = staggered[staggered["model"] == pick]
mine = holdout[holdout["model"] == pick]
level = pd.concat(
    [iv.levels(wide, origins(staggered=True)), iv.levels(wide, sorted(mine["origin"].unique()))]
)
checked = {}
for name, is_total in (("products", False), ("total", True)):
    q = iv.fit(staggered[(staggered["series"] == TOTAL) == is_total], level)
    checked[name] = iv.apply(mine[(mine["series"] == TOTAL) == is_total], level, q)
pd.concat({k: iv.coverage(v) for k, v in checked.items()}).round(3)

,,coverage,width,rows
products,0,0.810,2.308,10200
total,0,0.833,0.926,102


In [6]:
per_product = iv.coverage(checked["products"], "series")["coverage"]
per_product.describe().round(2)

count    100.00
mean       0.81
std        0.11
min        0.25
25%        0.77
50%        0.84
75%        0.88
max        0.96
Name: coverage, dtype: float64

In [7]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.8), dpi=150)
for model in ("weekday_mean_4", "ets", pick):
    m = products[products["model"] == model]
    g = m.assign(e=(m["forecast"] - m["actual"]).abs()).groupby("origin")
    w = g["e"].sum() / g["actual"].sum()
    left.plot(w.index, w.to_numpy(), marker="o", ms=3, lw=1.8, color=COLORS.get(model, MUTED),
              label=LABELS.get(model, model))  # fmt: skip
left.set_ylabel("wMAPE, 100 products pooled", color=MUTED)
left.set_ylim(0, None)
left.legend(frameon=False, fontsize=8, loc="lower left")
left.tick_params(axis="x", labelrotation=30)
m = products[products["model"] == pick]
g = m.groupby("origin")
bias = g["forecast"].sum() / g["actual"].sum() - 1
right.bar(
    np.arange(len(bias)),
    bias.to_numpy(),
    color=[COLORS[pick] if b < 0 else "#eb6834" for b in bias],
)
right.axhline(0, color=INK, lw=1)
right.set_xticks(np.arange(len(bias)), [d.strftime("%d %b") for d in bias.index], rotation=30)
right.set_ylabel("units forecast / sold - 1", color=MUTED)
for ax in (left, right):
    ax.grid(axis="y", color=GRID)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(
    "Holdout: LightGBM beats ETS at 7 of 8 origins, but lags the Christmas peak",
    x=0.01, ha="left", color=INK, fontsize=11,
)  # fmt: skip
fig.tight_layout()
fig.savefig("reports/figures/holdout.png", bbox_inches="tight")